In [1]:
import os

from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from dotenv import load_dotenv
from pydantic import BaseModel

/Users/biratpoudel/Desktop/blys/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
load_dotenv(dotenv_path=".env.local")

openai_api_key = os.getenv("OPENAI_API_KEY")

In [3]:
non_streaming_model_openai = ChatOpenAI(
    api_key=openai_api_key, model="gpt-4o", temperature=0, streaming=False
)

In [4]:
process_customer_query_prompt = """You are an expert Customer Support Service Staff. You can understand and respond to customer 
queries related to booking, cancellation, and pricing. You can also take action on behalf of the client according to the requests, 
for example, reschedule booking at a given date and time. Your responses must be helpful, concise, and proactive."""

In [5]:
class CustomerQueryResponse(BaseModel):
    intent: str
    response: str

In [8]:
def process_customer_query(customer_query: str):
    prompt = ChatPromptTemplate.from_messages(
        [("system", process_customer_query_prompt), ("user", "{customer_query}")]
    )

    runnable = prompt | non_streaming_model_openai.with_structured_output(
        schema=CustomerQueryResponse
    )

    return runnable.invoke({"customer_query": customer_query}).model_dump()

In [9]:
if __name__ == "__main__":
    query_1 = "Can I reschedule my booking?"
    result_1 = process_customer_query(query_1)
    print(result_1)

    query_2 = "I'd like to change my massage to 30 Mar 2025 at 10 am."
    result_2 = process_customer_query(query_2)
    print(result_2)

{'intent': 'reschedule_booking', 'response': "Yes, you can reschedule your booking. Please provide the new date and time you would like to reschedule to, and I'll take care of it for you."}
{'intent': 'reschedule_booking', 'response': 'Reschedule the massage appointment to 30 March 2025 at 10:00 AM.'}
